# 04 JUNE 2026

In [ ]:
import pandas as pd
import numpy as np
import re
import emoji
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
import torch
from torch import nn,optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

path = r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\train.csv"

df_train = pd.read_csv(path)
df_train.drop(['id'], axis=1,inplace=True)

lemmatizer = WordNetLemmatizer()
sw = stopwords.words('english')
    
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|https\S+|www\S+','',text)
    text = re.sub(r"<.*?>",'',text)
    text = re.sub(r'@\w+|#\w+','',text)
    text = re.sub(r'(.)\1{2,}',r'\1\1',text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = emoji.replace_emoji(text, '')
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in sw]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

df_train['cleaned_comment'] = df_train['comment_text'].apply(clean_text)
X = df_train['cleaned_comment']
df_train['toxicity_ind'] = df_train[['toxic',
                         'severe_toxic',
                         'obscene',
                         'threat',
                         'insult',
                         'identity_hate']].sum(axis=1)
df_train['toxicity_ind'] = df_train['toxicity_ind'].apply(lambda x: 1 if x > 0 else 0)

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = 30000,
    output_sequence_length =100,
    output_mode = 'int' 
)
vectorizer.adapt(X.values)
vectorized_text = vectorizer(X.values)
vocab_size = len(vectorizer.get_vocabulary())

class BiLSTM(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size,output_size):
        super(BiLSTM,self).__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=embed_size,padding_idx=0)
        # Bidirectional LSTM
        self.lstm = nn.LSTM(embed_size,hidden_size,num_layers=1,batch_first=True,bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_size * 2,output_size)
        
    def forward(self,X):
        X = self.embedding(X)
        out,(ht, ct) =self.lstm(X)
        forward_hidden = ht[-2]
        backward_hidden = ht[-1]
        # Concatenate forward & backward hidden states
        out = torch.cat((forward_hidden, backward_hidden), dim=1)
        out = self.dropout(out)
        out = self.fc(out)
        return out

X_train, X_val, y_train, y_val = train_test_split(
    vectorized_text.numpy(),
    df_train['toxicity_ind'].values,
    test_size=0.2,
    random_state=42,
    stratify=df_train['toxicity_ind']
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.long)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)
y_val = torch.tensor(y_val, dtype=torch.float32).reshape(-1,1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(
    val_dataset,
    batch_size=128
)

model = BiLSTM(vocab_size, embed_size=64, hidden_size=128, output_size=1)

positive_count = sum(df_train['toxicity_ind'])
negative_count = len(df_train) - positive_count
pos_weight = torch.tensor([negative_count / positive_count])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=1e-5)
best_val_loss = float('inf')
patience = 3
counter = 0

for epoch in range(10):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        output = model(batch_X)
        loss = criterion(output, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    #print(f"Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")

    val_loss = 0
    all_preds = []
    all_labels = []
    model.eval()
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            #threshold = 0.3
            #preds = (probs > threshold).float()
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()

    print(
            f"Epoch {epoch+1} "
            f"Train Loss:{avg_loss:.4f} "
            f"Val Loss:{avg_val_loss:.4f}"
        )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(
                        {
                            'model_state_dict': model.state_dict(),
                            'vocab_size': vocab_size,
                            'embed_size': 64,
                            'hidden_size': 128,
                            'output_size': 1
                        },
                        "best_bilstm_model.pth"
                    )
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

    for threshold in [0.3, 0.4, 0.5, 0.6]:
        preds = (all_preds > threshold).astype(int)
        print(f"\nThreshold = {threshold}")
        print(
            classification_report(
                all_labels,
                preds,
                digits=4
            )
        )
        print(
            "Confusion Matrix:\n",
            confusion_matrix(all_labels, preds)
        )

Epoch 1 Train Loss:0.7680 Val Loss:0.5243

Threshold = 0.3
              precision    recall  f1-score   support

         0.0     0.9868    0.8299    0.9016     28670
         1.0     0.3750    0.9017    0.5297      3245

    accuracy                         0.8372     31915
   macro avg     0.6809    0.8658    0.7156     31915
weighted avg     0.9246    0.8372    0.8637     31915

Confusion Matrix:
 [[23793  4877]
 [  319  2926]]

Threshold = 0.4
              precision    recall  f1-score   support

         0.0     0.9840    0.8844    0.9315     28670
         1.0     0.4608    0.8730    0.6032      3245

    accuracy                         0.8832     31915
   macro avg     0.7224    0.8787    0.7674     31915
weighted avg     0.9308    0.8832    0.8982     31915

Confusion Matrix:
 [[25355  3315]
 [  412  2833]]

Threshold = 0.5
              precision    recall  f1-score   support

         0.0     0.9806    0.9204    0.9496     28670
         1.0     0.5441    0.8394    0.6603 

# 05 JUNE 2026

In [24]:
import pandas as pd
import numpy as np
import re
import emoji
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
import torch
from torch import nn,optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

path = r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\train.csv"

df_train = pd.read_csv(path)
df_train.drop(['id'], axis=1,inplace=True)

lemmatizer = WordNetLemmatizer()
sw = stopwords.words('english')
    
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|https\S+|www\S+','',text)
    text = re.sub(r"<.*?>",'',text)
    text = re.sub(r'@\w+|#\w+','',text)
    text = re.sub(r'(.)\1{2,}',r'\1\1',text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = emoji.replace_emoji(text, '')
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in sw]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

df_train['cleaned_comment'] = df_train['comment_text'].apply(clean_text)
X = df_train['cleaned_comment']
df_train['toxicity_ind'] = df_train[['toxic',
                         'severe_toxic',
                         'obscene',
                         'threat',
                         'insult',
                         'identity_hate']].sum(axis=1)
df_train['toxicity_ind'] = df_train['toxicity_ind'].apply(lambda x: 1 if x > 0 else 0)

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = 30000,
    output_sequence_length =100,
    output_mode = 'int' 
)
vectorizer.adapt(X.values)
vectorized_text = vectorizer(X.values)
vocab_size = len(vectorizer.get_vocabulary())

class BiLSTM(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size,output_size):
        super(BiLSTM,self).__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=embed_size,padding_idx=0)
        # Bidirectional LSTM
        self.lstm = nn.LSTM(embed_size,hidden_size,num_layers=1,batch_first=True,bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_size * 2,output_size)
        
    def forward(self,X):
        X = self.embedding(X)
        out,(ht, ct) =self.lstm(X)
        forward_hidden = ht[-2]
        backward_hidden = ht[-1]
        # Concatenate forward & backward hidden states
        out = torch.cat((forward_hidden, backward_hidden), dim=1)
        out = self.dropout(out)
        out = self.fc(out)
        return out

X_train, X_val, y_train, y_val = train_test_split(
    vectorized_text.numpy(),
    df_train['toxicity_ind'].values,
    test_size=0.2,
    random_state=42,
    stratify=df_train['toxicity_ind']
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.long)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)
y_val = torch.tensor(y_val, dtype=torch.float32).reshape(-1,1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(
    val_dataset,
    batch_size=128
)

model = BiLSTM(vocab_size, embed_size=64, hidden_size=128, output_size=1)

positive_count = sum(df_train['toxicity_ind'])
negative_count = len(df_train) - positive_count
pos_weight = torch.tensor([negative_count / positive_count])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=1e-5)
best_val_loss = float('inf')
patience = 3
counter = 0

for epoch in range(10):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        output = model(batch_X)
        loss = criterion(output, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    #print(f"Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")

    val_loss = 0
    all_preds = []
    all_labels = []
    model.eval()
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            #threshold = 0.3
            #preds = (probs > threshold).float()
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()

    print(
            f"Epoch {epoch+1} "
            f"Train Loss:{avg_loss:.4f} "
            f"Val Loss:{avg_val_loss:.4f}"
        )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(
                        {
                            'model_state_dict': model.state_dict(),
                            'vocab_size': vocab_size,
                            'embed_size': 64,
                            'hidden_size': 128,
                            'output_size': 1
                        },
                        "best_bilstm_model.pth"
                    )
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

    

Epoch 1 Train Loss:0.7454 Val Loss:0.5456
Epoch 2 Train Loss:0.4315 Val Loss:0.4314
Epoch 3 Train Loss:0.3332 Val Loss:0.4067
Epoch 4 Train Loss:0.2711 Val Loss:0.4295
Epoch 5 Train Loss:0.2285 Val Loss:0.4359
Epoch 6 Train Loss:0.2068 Val Loss:0.5242
Early stopping


# Save & Load model/Vectorizer for prediction

In [25]:
vocab = vectorizer.get_vocabulary()

import pickle

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# Predict text using model

In [27]:
df_test = pd.read_csv(r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\test.csv")
df_test = df_test.sample(5)
df_test['cleaned_comment'] = df_test['comment_text'].apply(clean_text)
vector_test = vectorizer(df_test['cleaned_comment'].values)
labels = []
scores = []

for sentence in df_test['cleaned_comment']:
    label, score = predict_toxicity(sentence)
    labels.append(label)
    scores.append(score)
df_test.drop(['cleaned_comment'], axis=1,inplace=True)
# Add predictions to dataframe
df_test['toxicity_label'] = labels
df_test['toxicity_probability'] = scores
df_test

,id,comment_text,toxicity_label,toxicity_probability
29606,313ca818efc0718d,== Thanks much == \n\n \n Thank you for your...,Non-Toxic,0.000867
57618,5fda63de16c5663e,==Thanks for fixing Peter Thompson== \n Seesdi...,Non-Toxic,0.128098
96656,a1483755402052e5,:There is a reason why everyone is so keen not...,Non-Toxic,0.005169
93894,9c9ab9d7d5c8f3ee,*** What policy says that long-term usage trum...,Non-Toxic,0.000400
142950,eeee28647ef773fe,this is absolute crap.,Toxic,0.995792


In [39]:
with open("C:\\Users\\nisha\\OneDrive\\Desktop\\Nisha\\GUVI\\MiniProject\\CommentToxicity\\Model\\vocab.pkl", "rb") as f:
    vectorizer_data = pickle.load(f)
    
vectorizer = tf.keras.layers.TextVectorization(
    output_mode='int',
    output_sequence_length=100   # Use the same value as during training
)

vectorizer.set_vocabulary(vocab)

sample = ["you are stupid"]

vectorized = vectorizer(sample)

print(vectorized)
print(vectorized.numpy())

tf.Tensor(
[[  1   1 460   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]], shape=(1, 100), dtype=int64)
[[  1   1 460   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]]


In [35]:
with open("C:\\Users\\nisha\\OneDrive\\Desktop\\Nisha\\GUVI\\MiniProject\\CommentToxicity\\Model\\vocab.pkl", "rb") as f:
    vectorizer_data = pickle.load(f)

print(type(vectorizer_data))
print(vectorizer_data[:5])  # if it's a list

<class 'list'>
['', '[UNK]', np.str_('article'), np.str_('page'), np.str_('wikipedia')]


In [28]:
df_test = pd.read_csv(r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\test.csv")
df_test = df_test.sample(10)
df_test.to_csv(r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\sample_test.csv", index=False)